In [5]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import urllib
import requests
import json
import pymysql

In [8]:
api_key = "53474b575a6b676d35314b4b536259"
type = 'json'
start = 1
stop = 5
url = f"http://openapi.seoul.go.kr:8088/{api_key}/{type}/tbCycleStationInfo/{start}/{stop}/"
response = requests.get(url)

In [9]:
result_dict = response.json()
result_df = pd.DataFrame(result_dict['stationInfo']['row'])

result_df

,STA_LOC,RENT_ID,RENT_NO,RENT_NM,RENT_ID_NM,HOLD_NUM,STA_ADD1,STA_ADD2,STA_LAT,STA_LONG,START_INDEX,END_INDEX,RNUM
0,마포구,ST-10,00108,서교동 사거리,108. 서교동 사거리,12,서울특별시 마포구 양화로 93,427,37.55274582,126.91861725,0,0,1
1,광진구,ST-100,00503,더샵스타시티 C동 앞,503. 더샵스타시티 C동 앞,15,서울특별시 광진구 아차산로 262,더샵스타시티 C동 앞,37.53666687,127.07359314,0,0,2
2,양천구,ST-1000,00729,서부식자재마트 건너편,729. 서부식자재마트 건너편,10,서울특별시 양천구 신정동 236,서부식자재마트 건너편,37.51037979,126.86679840,0,0,3
3,양천구,ST-1002,00731,서울시 도로환경관리센터,731. 서울시 도로환경관리센터,10,서울특별시 양천구 목동동로 316-6,서울시 도로환경관리센터,37.52989960,126.87654114,0,0,4
4,양천구,ST-1003,00732,신월중학교,732. 신월중학교,10,서울특별시 양천구 화곡로 59,신월동 이마트,37.53955078,126.82830048,0,0,5


In [13]:
conn = pymysql.connect(
    host = '127.0.0.1',
    user = 'humanda6',
    passwd = 'humanda6',
    database = 'data_repo'
)
cur = conn.cursor()

def sql_get(sql):
    cur.execute(sql)
    return cur.fetchall()

def sql_set(sql):
    cur.execute(sql)
    conn.commit()

sql_set('''DROP TABLE IF EXISTS rental_station''')
sql_set("""CREATE TABLE rental_station
         (
            STA_LOC varchar(20) not null,
            RENT_ID varchar(20) not null,
            RENT_NO varchar(20) not null primary key,
            RENT_NM varchar(50) not null,
            RENT_ID_NM varchar(50) not null,
            HOLD_NUM int not null,
            STA_ADD1 varchar(100) not null,
            STA_ADD2 varchar(100) null,
            STA_LAT decimal(11,8) not null,
            STA_LONG decimal(11,8) not null
         )""")

cur.executemany("""INSERT INTO rental_station VALUES(%s, %s, %s, %s, %s, %s, %s, %s, %s, %s)""",
                result_df.values[:,:-3].tolist()
             )
conn.commit()

df = pd.DataFrame( sql_get('''SELECT * FROM rental_station'''), columns=result_df.columns[:-3] ) 

cur.close()
conn.close()

df

,STA_LOC,RENT_ID,RENT_NO,RENT_NM,RENT_ID_NM,HOLD_NUM,STA_ADD1,STA_ADD2,STA_LAT,STA_LONG
0,마포구,ST-10,00108,서교동 사거리,108. 서교동 사거리,12,서울특별시 마포구 양화로 93,427,37.55274582,126.91861725
1,광진구,ST-100,00503,더샵스타시티 C동 앞,503. 더샵스타시티 C동 앞,15,서울특별시 광진구 아차산로 262,더샵스타시티 C동 앞,37.53666687,127.07359314
2,양천구,ST-1000,00729,서부식자재마트 건너편,729. 서부식자재마트 건너편,10,서울특별시 양천구 신정동 236,서부식자재마트 건너편,37.51037979,126.86679840
3,양천구,ST-1002,00731,서울시 도로환경관리센터,731. 서울시 도로환경관리센터,10,서울특별시 양천구 목동동로 316-6,서울시 도로환경관리센터,37.52989960,126.87654114
4,양천구,ST-1003,00732,신월중학교,732. 신월중학교,10,서울특별시 양천구 화곡로 59,신월동 이마트,37.53955078,126.82830048
